<div style="padding: 28px; border-radius: 18px; background: linear-gradient(120deg, #07111f 0%, #0e2940 45%, #0f766e 100%); color: #f8fafc; margin-bottom: 14px;">
  <div style="font-size: 12px; text-transform: uppercase; opacity: 0.85;">NeMo Evaluator SDK</div>
  <h1 style="margin: 8px 0 10px; font-weight: 800; line-height: 1.1;">Evaluate a Coding Agent with NeMo Fabric</h1>
  <div style="font-size: 15px; opacity: 0.92; max-width: 720px;">
    Build an <b>agentic evaluation</b> from scratch: define a small suite of realistic coding tasks
    (fix a bug, write tests, write docs), run them against the <b>Codex</b> coding agent driven by
    <b>NeMo Fabric</b>, score each against <b>held-out</b> ground truth the agent can't touch, and roll
    heterogeneous per-task metrics up into one comparable <b>correctness</b> score.
  </div>
</div>

## What is an *agentic* evaluation?

A classic evaluation scores a **model's answer** against a reference. An *agentic* evaluation scores an
**agent's behavior on a task** — the agent reads files, runs commands, edits a workspace — and what we
judge is the *result of that work*.

```
AgentEvalTask   →   run against a target   →   AgentEvalTrial(s)   →   Metrics   →   AgentEvalResult
(intent, inputs,     (Codex, driven by         (what the agent did:    (score each      (trials, scores,
 reference, metrics,  NeMo Fabric)              output + workspace      trial vs.        summary)
 views)                                         + trajectory)           held-out truth)
```

- **`AgentEvalTask`** — one unit of work: an `intent`, `inputs` (files seeded into the workspace), a
  grader-only **`reference`** (held-out ground truth), the `metrics` that score it, and optional `views`.
- **Target** — what runs the task. We use **`FabricAgentRuntime`**: NeMo Fabric drives the Codex CLI,
  captures the final workspace **and** the agent's execution trajectory.
- **`AgentEvalTrial`** — the captured run: the agent's output plus **evidence** (its final `workspace`
  and its ATIF `trace`).
- **Metric** — scores one trial. It can open the workspace evidence (does a file exist? do the tests
  pass?) and read the task's held-out `reference` — which the agent never saw.
- **View** — maps a task's *own* metrics into a shared, task-agnostic score. "Correct" means different
  things per task (tests pass vs. a docstring exists); a view named `correctness` lets us compare and
  roll them up. **This is how you avoid hand-writing pass/fail logic.**
- **`AgentEvalResult`** — trials, per-metric scores, and an aggregated summary (including `view.*`
  rollups).

> This uses **`AgentEvaluator`**, the SDK-native path that runs locally and returns results in-process.
> The evaluator *measures*; deciding pass/fail thresholds is an application concern.

## Notebook map

1. [Install & prerequisites](#prereqs)
2. [Set up a workspace](#setup)
3. [Metrics: how a trial gets scored](#metrics)
4. [Define the task suite](#tasks) — [fix a bug](#task-fix-bug) · [write tests](#task-write-tests) · [write docs](#task-write-docs)
5. [Pick the target: Codex via NeMo Fabric](#target)
6. [Run the evaluation](#run)
7. [Read the results](#results)
8. [Where to go next](#next)

<a id="prereqs"></a>
## 1. Install & prerequisites

This tutorial evaluates **Codex driven by NeMo Fabric**. Fabric is a native (Rust) component, so —
unlike a pure-Python SDK — this notebook does **not** run from a plain `pip install`. You need a real
toolchain:

- **NeMo Evaluator SDK** — `pip install "nemo-platform-sdk[nemo-evaluator-sdk]"` and `pip install pytest` (some metrics run the agent's tests).
- **NeMo Fabric** (native) with the coding-agent + trajectory extras — `nemo-fabric[codex,relay]` —
  plus a **checkout of the NeMo-Fabric repo**: the Codex adapter registry (`adapters/codex-cli`) is
  resolved relative to it. The repo's `script/dev-install-fabric.sh` installs `nemo-fabric[codex,relay]`
  and builds the `nemo-relay` gateway for you.
- **Codex CLI** on your `PATH` (`npm install -g @openai/codex`) + auth (`codex login`, or set `OPENAI_API_KEY`).
- **`nemo-relay` gateway** binary on your `PATH` (built by the script above) — required for ATIF
  trajectory capture.

Set two environment variables before launching (both read from the environment, never written here):

- **`NEMO_FABRIC_REPO`** — path to your NeMo-Fabric checkout (Fabric resolves the adapter registry
  from it).
- **`NVIDIA_BUILD_API_KEY`** — a build.nvidia.com key for the `write-docs` LLM judge. The cell below
  prompts for it if it isn't already set.

Everything else — defining tasks and metrics, reading results — is plain Python you can read through.

In [ ]:
import getpass
import os

# Prompt for the judge's API key unless it's already in the environment. getpass masks the input, so
# the key never lands in the notebook or its saved output.
if not os.environ.get("NVIDIA_BUILD_API_KEY"):
    os.environ["NVIDIA_BUILD_API_KEY"] = getpass.getpass("build.nvidia.com API key (NVIDIA_BUILD_API_KEY): ")

# Fabric resolves the Codex adapter registry relative to a NeMo-Fabric checkout. It's a path, not a
# secret, so prompt with plain input() (not getpass) and expand ~ if the environment doesn't set it.
if not os.environ.get("NEMO_FABRIC_REPO"):
    os.environ["NEMO_FABRIC_REPO"] = os.path.expanduser(
        input("Path to your NeMo-Fabric checkout (NEMO_FABRIC_REPO): ").strip()
    )
if not os.environ["NEMO_FABRIC_REPO"]:
    raise RuntimeError("NEMO_FABRIC_REPO is required — set it to your NeMo-Fabric checkout.")

<a id="setup"></a>
## 2. Set up a workspace

Everything runs locally. We pick an output directory for the run bundle (trials, scores, summary, and
an HTML dashboard) and import the pieces we'll use.

In [ ]:
import hashlib
import os
import re
import sys
from pathlib import Path
from tempfile import mkdtemp

from nemo_evaluator_sdk.agent_eval.evaluator import AgentEvaluator
from nemo_evaluator_sdk.agent_eval.metrics import AgentPhaseSuccessMetric
from nemo_evaluator_sdk.agent_eval.tasks import (
    AgentEvalRunConfig,
    AgentEvalTask,
    AgentEvalTaskset,
    SemanticReducer,
    SemanticView,
    ViewSignal,
)
from nemo_evaluator_sdk.metrics.protocol import MetricInput, MetricOutput, MetricOutputSpec, MetricResult

OUTPUT_DIR = Path(mkdtemp(prefix="agent-eval-"))
print("Run bundle will be written to:", OUTPUT_DIR)


def inline(text: str) -> dict:
    """An inline seed file — contents carried in the task. Equivalent to passing the bare string."""
    return {"kind": "inline", "content": text}

<a id="metrics"></a>
## 3. Metrics: how a trial gets scored

A **metric** implements a tiny protocol — a `type` name, an `output_spec()` declaring the named values
it emits, and an async `compute_scores(input)` returning them for one trial. The metric receives a
`MetricInput` whose `candidate` carries the agent's `output_text` **and its evidence** — for a Fabric
coding trial that includes the final **`workspace`** directory (and the ATIF **`trace`**). `input.row`
carries the task's grader-only **`reference`** at `input.row.data["reference"]`. That combination is
what makes *coding* work scorable against ground truth the agent never saw.

<div style="padding: 16px 20px; border-left: 5px solid #f59e0b; background: #fffbeb; border-radius: 8px; color: #451a03;">
<b>⚠️ Grade only with artifacts the agent cannot edit.</b><br/>
If the tests that score a "fix the bug" task live in the agent's own writable workspace, a coding
agent can — and, optimizing for reward, sometimes will — <i>edit the tests</i> instead of fixing the
bug. The evaluation then measures nothing. Keep ground truth <b>held out</b>: put it in the task's
<code>reference</code> (grader-only, never seeded into the workspace, never shown to the agent). A
metric then either <b>overlays</b> it into a throwaway copy before running a verifier, or <b>checksums</b>
a file the agent was told not to touch. The metrics below do both.
</div>

We define three metrics. `AgentPhaseSuccessMetric` (did the agent finish cleanly?) ships with the SDK;
the other two are small classes you can read through.

`PytestResults` leans on **`LocalFilesystemEvidence.run_verifier`**, which copies the workspace to a
throwaway directory and runs a command there (so scoring never mutates the trial's evidence). Its
`overlay_files` argument writes trusted files *over* that copy first — that's how the held-out tests
get in without ever being in the agent's workspace.

In [ ]:
class WorkspaceFileContains:
    """True when `path` exists in the agent's workspace and includes the given substring (case-insensitive)."""

    def __init__(self, *, name: str, path: str, contains: str) -> None:
        self._name = name
        self._path = path
        self._contains = contains

    @property
    def type(self) -> str:
        return self._name

    def output_spec(self) -> list[MetricOutputSpec]:
        return [MetricOutputSpec.boolean("present")]

    async def compute_scores(self, input: MetricInput) -> MetricResult:
        present = False
        evidence = input.candidate.evidence
        if evidence is not None and evidence.get("workspace") is not None:
            ws = await evidence.filesystem("workspace")
            if await ws.exists(self._path):
                present = self._contains.lower() in (await ws.read_text(self._path)).lower()
        return MetricResult(outputs=[MetricOutput(name="present", value=present)])


class WorkspaceFileUnchanged:
    """True when a workspace file byte-matches the held-out reference — i.e. the agent left it alone.

    For a "write tests" task the deliverable *is* the tests, so they can't be held out. The reward
    hack instead is editing the code under test so weak tests pass. We keep the authoritative source in
    ``reference`` and checksum the agent's copy against it, turning "don't touch the module" into an
    explicit score rather than trusting the agent.
    """

    def __init__(self, *, name: str, path: str, reference_key: str) -> None:
        self._name = name
        self._path = path
        self._reference_key = reference_key

    @property
    def type(self) -> str:
        return self._name

    def output_spec(self) -> list[MetricOutputSpec]:
        return [MetricOutputSpec.boolean("unchanged")]

    async def compute_scores(self, input: MetricInput) -> MetricResult:
        unchanged = False
        expected = (input.row.data.get("reference") or {}).get(self._reference_key, {}).get(self._path)
        evidence = input.candidate.evidence
        if expected is not None and evidence is not None and evidence.get("workspace") is not None:
            ws = await evidence.filesystem("workspace")
            if await ws.exists(self._path):
                actual = await ws.read_text(self._path)
                unchanged = _sha256(actual) == _sha256(expected)
        return MetricResult(outputs=[MetricOutput(name="unchanged", value=unchanged)])


def _sha256(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def _parse_pytest_counts(text: str) -> tuple[int, int]:
    """Pull (passed, failed+errored) counts out of pytest's summary line."""

    def n(word: str) -> int:
        match = re.search(rf"(\d+) {word}", text)
        return int(match.group(1)) if match else 0

    return n("passed"), n("failed") + n("error")


class PytestResults:
    """Run pytest against the agent's solution and report several outputs, not just a boolean.

    If ``overlay_reference_key`` is set, the files under ``reference[key]`` are staged *over* a
    throwaway copy of the workspace before pytest runs — held-out tests the agent never had and could
    not edit. Otherwise the agent's own workspace (its files and any tests it wrote) is run as-is.
    """

    def __init__(self, *, overlay_reference_key: str | None = None) -> None:
        self._overlay_reference_key = overlay_reference_key

    @property
    def type(self) -> str:
        return "pytest_results"

    def output_spec(self) -> list[MetricOutputSpec]:
        return [
            MetricOutputSpec.boolean("all_passed"),
            MetricOutputSpec.continuous_score("pass_rate"),
            MetricOutputSpec.discrete_score("num_run"),
            MetricOutputSpec.discrete_score("num_failed"),
        ]

    async def compute_scores(self, input: MetricInput) -> MetricResult:
        passed, failed, ok = 0, 0, False
        overlay = {}
        if self._overlay_reference_key:
            overlay = (input.row.data.get("reference") or {}).get(self._overlay_reference_key, {})
        evidence = input.candidate.evidence
        if evidence is not None and evidence.get("workspace") is not None:
            ws = await evidence.filesystem("workspace")
            result = await ws.run_verifier(
                [sys.executable, "-m", "pytest", "-q"],
                overlay_files=overlay or None,
                timeout_s=120,
            )
            passed, failed = _parse_pytest_counts(result.stdout + result.stderr)
            ok = result.ok
        num_run = passed + failed
        return MetricResult(
            outputs=[
                MetricOutput(name="all_passed", value=ok and num_run > 0),
                MetricOutput(name="pass_rate", value=(passed / num_run) if num_run else 0.0),
                MetricOutput(name="num_run", value=num_run),
                MetricOutput(name="num_failed", value=failed),
            ]
        )

<a id="tasks"></a>
## 4. Define the task suite

Three coding tasks. Each seeds starter files into the agent's workspace via `inputs["files"]` — a
`{path: source}` map the runner stages before the agent starts (a `source` is `inline` contents here;
it can also be a local `path` or a stored `fileset` reference, picked by a `kind` field — a bare
string is shorthand for inline text). What the agent is graded against lives in **`reference`**, which
is *never* seeded into the workspace. Each task attaches **its own** metrics and a `correctness`
**view** that maps those task-specific metrics onto one shared pass/fail score.

Note how each `ViewSignal` refers to a metric by its `.type` rather than a hand-typed string — the
reference stays correct if a metric's type name ever changes.

In [ ]:
# Metric instances. fix-bug overlays held-out tests; write-tests runs the agent's own tests but
# checksums the module it must not modify. Referencing `metric.type` in views keeps the wiring honest.
phase_success = AgentPhaseSuccessMetric()
pytest_with_held_out_tests = PytestResults(overlay_reference_key="tests")
pytest_agent_tests = PytestResults()

<a id="task-fix-bug"></a>
### 4a. Fix a bug

We seed a buggy `calculator.py` — **and nothing else**. The test suite is *held out* in `reference`
and overlaid into a throwaway copy at scoring time, so the agent physically cannot edit the tests that
grade it; it can only fix the code.
Scored by: `pytest_results` (do the held-out tests pass, and at what rate) and `AgentPhaseSuccessMetric`
(did the agent exit cleanly). `correctness` = *agent finished* **and** *all held-out tests pass*.

In [ ]:
BUGGY_CALCULATOR = """def add(a, b):
    # BUG: subtracts instead of adding
    return a - b
"""

CALCULATOR_TESTS = """from calculator import add


def test_add():
    assert add(2, 3) == 5
    assert add(-1, 1) == 0
"""

fix_bug = AgentEvalTask(
    id="fix-bug",
    intent="Fix the bug in calculator.py so that add() returns the sum and the hidden tests pass.",
    inputs={
        "instruction": (
            "calculator.py has a bug: add(a, b) subtracts instead of adding. Fix calculator.py so that "
            "add() returns the sum. You are graded by a hidden test suite you cannot see or edit."
        ),
        # Only the buggy source is seeded. The test file is NOT here — it is held out in `reference`.
        "files": {"calculator.py": inline(BUGGY_CALCULATOR)},
    },
    # Grader-only ground truth: overlaid by PytestResults at scoring time, never seen by the agent.
    reference={"tests": {"test_calculator.py": CALCULATOR_TESTS}},
    metrics=[phase_success, pytest_with_held_out_tests],
    views={
        "correctness": SemanticView(
            reducer=SemanticReducer.ALL,
            signals=[
                ViewSignal(metric=phase_success.type, output="agent_phase_success"),
                ViewSignal(metric=pytest_with_held_out_tests.type, output="all_passed"),
            ],
        )
    },
)

<a id="task-write-tests"></a>
### 4b. Write tests

We seed a correct `stringutils.py` so the agent can read it. **The agent must write a passing pytest
file for it.** Here the deliverable *is* the tests, so they can't be held out — but the reward hack is
editing the module so weak tests pass. So we keep the authoritative `stringutils.py` in `reference`
and **checksum** the agent's copy against it: `impl_unchanged` is a first-class signal, and the
`correctness` view fails if the agent touched the module. Scored by: `wrote_test_file` (did a
`test_stringutils.py` referencing `slugify` appear), `impl_unchanged` (module left alone), and
`pytest_results` (the agent's tests pass). A different metric set from the bug task.

In [ ]:
STRINGUTILS = """import re


def slugify(text: str) -> str:
    \"\"\"Lowercase text and turn runs of non-alphanumerics into single hyphens.\"\"\"
    return re.sub(r"[^a-z0-9]+", "-", text.lower()).strip("-")
"""

wrote_test_file = WorkspaceFileContains(name="wrote_test_file", path="test_stringutils.py", contains="slugify")
impl_unchanged = WorkspaceFileUnchanged(name="impl_unchanged", path="stringutils.py", reference_key="protected")

write_tests = AgentEvalTask(
    id="write-tests",
    intent="Write pytest tests for the slugify() function in stringutils.py.",
    inputs={
        "instruction": (
            "Write a pytest file named test_stringutils.py that imports slugify from stringutils and "
            "covers a few cases. The tests must pass. Do not modify stringutils.py."
        ),
        "files": {"stringutils.py": inline(STRINGUTILS)},
    },
    # The authoritative module the agent must not touch; impl_unchanged checksums against it.
    reference={"protected": {"stringutils.py": STRINGUTILS}},
    metrics=[wrote_test_file, impl_unchanged, pytest_agent_tests],
    views={
        "correctness": SemanticView(
            reducer=SemanticReducer.ALL,
            signals=[
                ViewSignal(metric=wrote_test_file.type, output="present"),
                ViewSignal(metric=impl_unchanged.type, output="unchanged"),
                ViewSignal(metric=pytest_agent_tests.type, output="all_passed"),
            ],
        )
    },
)

<a id="task-write-docs"></a>
### 4c. Write docs

We seed an undocumented `widget.py`. **The agent must add a docstring and a README.**

Presence is easy to check deterministically — but *is the documentation any good?* That's subjective,
so we add an **LLM-as-judge** metric alongside the deterministic ones. Deterministic checks feed
`correctness` (did the files appear); the judge scores `doc_quality` against a **rubric** (it picks a
label like `excellent`/`good`/…, which maps to a 1–5 value) for how clear and complete they are.

In [ ]:
from nemo_evaluator_sdk.metrics.llm_judge import LLMJudgeMetric
from nemo_evaluator_sdk.metrics.protocol import CandidateOutput, DatasetRow
from nemo_evaluator_sdk.values import InferenceParams, Model, SecretRef
from nemo_evaluator_sdk.values.scores import Rubric, RubricScore

# The judge model's API key is injected via this env var — it never appears in the notebook.
JUDGE_API_KEY_ENV = "NVIDIA_BUILD_API_KEY"


class LlmDocReview:
    """LLM-as-judge over the docs the agent produced.

    Composition, not inheritance: this metric *holds* an LLMJudgeMetric (which owns the model call and
    the structured-JSON parsing) and adds only what is specific here — pulling the README and module
    source out of the workspace evidence and handing them to the judge to grade.
    """

    def __init__(self, judge: LLMJudgeMetric) -> None:
        self._judge = judge

    @property
    def type(self) -> str:
        return "doc_review"

    def output_spec(self) -> list[MetricOutputSpec]:
        return self._judge.output_spec()

    async def compute_scores(self, input: MetricInput) -> MetricResult:
        readme, module_source = "", ""
        evidence = input.candidate.evidence
        if evidence is not None and evidence.get("workspace") is not None:
            ws = await evidence.filesystem("workspace")
            if await ws.exists("README.md"):
                readme = await ws.read_text("README.md")
            if await ws.exists("widget.py"):
                module_source = await ws.read_text("widget.py")
        # Hand the artifacts to the judge as its item fields; the judge owns the model call.
        judge_input = MetricInput(
            row=DatasetRow(row_index=0, data={"readme": readme, "module_source": module_source}),
            candidate=CandidateOutput(output_text=readme),
        )
        return await self._judge.compute_scores(judge_input)


doc_judge = LlmDocReview(
    LLMJudgeMetric(
        model=Model(
            url="https://integrate.api.nvidia.com/v1/chat/completions",
            name=os.environ.get("JUDGE_MODEL", "nvidia/nvidia-nemotron-nano-9b-v2"),
            format="openai",
            # api_key_secret names the env var to read; model.api_key resolves os.environ[JUDGE_API_KEY_ENV].
            api_key_secret=SecretRef(JUDGE_API_KEY_ENV),
        ),
        scores=[
            RubricScore(
                name="doc_quality",
                description="Overall documentation quality: clarity, accuracy, and completeness.",
                # A rubric fits an LLM judge better than a bare 1-5 range: each level is an explicit,
                # describable bar. The judge returns a label; the parser maps it to `value`.
                rubric=[
                    Rubric(
                        label="excellent",
                        value=5,
                        description="Clear, accurate, and complete, with a helpful usage example.",
                    ),
                    Rubric(label="good", value=4, description="Mostly clear and complete; minor gaps or rough edges."),
                    Rubric(label="adequate", value=3, description="Understandable but missing detail or an example."),
                    Rubric(label="poor", value=2, description="Vague, partly inaccurate, or hard to follow."),
                    Rubric(label="missing", value=1, description="Essentially undocumented or incorrect."),
                ],
            )
        ],
        prompt_template={
            "messages": [
                {
                    "role": "system",
                    "content": "You are a senior engineer reviewing documentation. Respond with JSON only.",
                },
                {
                    "role": "user",
                    "content": (
                        "A developer documented this module:\n```python\n{{item.module_source}}\n```\n\n"
                        "and wrote this README:\n```markdown\n{{item.readme}}\n```\n\n"
                        "Rate the documentation on clarity, accuracy, and completeness with one of "
                        "these labels — excellent, good, adequate, poor, missing — then "
                        'return JSON only: {"doc_quality": "<label>"}.'
                    ),
                },
            ]
        },
        # nemotron-nano is a reasoning model: budget enough tokens to think *and* still emit the final
        # JSON (reasoning grows with the input, so leave comfortable headroom).
        inference=InferenceParams.model_validate({"temperature": 0.0, "max_tokens": 2048}),
    )
)

> The judge reads its API key from `NVIDIA_BUILD_API_KEY` (via `Model.api_key_secret`), so the key is
> injected from the environment and never appears in the notebook.

In [ ]:
UNDOCUMENTED_WIDGET = """class Widget:
    def __init__(self, label):
        self.label = label

    def render(self):
        return f"[{self.label}]"
"""

widget_has_docstring = WorkspaceFileContains(name="widget_has_docstring", path="widget.py", contains='"""')
readme_mentions_widget = WorkspaceFileContains(name="readme_mentions_widget", path="README.md", contains="widget")

write_docs = AgentEvalTask(
    id="write-docs",
    intent="Document the Widget class: add a module docstring and a README.",
    inputs={
        "instruction": (
            "Add a module-level docstring to widget.py describing the Widget class, and create a "
            "README.md that explains what a Widget is with a short usage example."
        ),
        "files": {"widget.py": inline(UNDOCUMENTED_WIDGET)},
    },
    metrics=[
        widget_has_docstring,
        readme_mentions_widget,
        doc_judge,  # LLM-as-judge: rates doc_quality 1-5 from the workspace files
    ],
    views={
        "correctness": SemanticView(
            reducer=SemanticReducer.ALL,
            signals=[
                ViewSignal(metric=widget_has_docstring.type, output="present"),
                ViewSignal(metric=readme_mentions_widget.type, output="present"),
            ],
        )
    },
)

The three tasks form one suite. Group them into an **`AgentEvalTaskset`** — a named, validated set
(task ids must be unique) with free-form metadata — so the suite travels as a single object you can
name, version, and hand to the evaluator.

In [ ]:
suite = AgentEvalTaskset(
    tasks=[fix_bug, write_tests, write_docs],
    metadata={"name": "coding-agent-correctness", "description": "Fix a bug, write tests, write docs."},
)
print("Suite:", ", ".join(t.id for t in suite.tasks))

<a id="target"></a>
## 5. Pick the target: Codex via NeMo Fabric

The **target** is what runs each task. We use **`FabricAgentRuntime`**: NeMo Fabric drives a harness
(here the Codex CLI, selected by `harness.adapter_id`) in a fresh per-task workspace, and captures
both the final workspace and the agent's execution **trajectory**. Each task's `inputs["files"]` are
seeded into its workspace, the harness runs there, and the final file tree is exposed as `workspace`
evidence (what the metrics above open). `capture_trajectory=True` additionally records the agent's
step-by-step actions as an ATIF `trace` (via the `nemo-relay` gateway).

The agent config is built from Fabric's own typed config objects (`FabricConfig`, `HarnessConfig`, …)
rather than a raw dict, so the harness/runtime/environment fields are checked as you write them.
`base_dir` points at your NeMo-Fabric checkout so Fabric can resolve the Codex adapter registry.

In [ ]:
from nemo_evaluator_sdk.agent_eval.runtimes.fabric.runtime import FabricAgentRuntime
from nemo_fabric import (  # ty: ignore[unresolved-import]
    EnvironmentConfig,
    FabricConfig,
    HarnessConfig,
    MetadataConfig,
    RuntimeConfig,
)

# A typed Fabric agent config: Codex CLI harness, one-shot text in / message out, local execution.
codex_via_fabric = FabricConfig(
    metadata=MetadataConfig(name="coding-agent-eval"),
    harness=HarnessConfig(
        adapter_id="nvidia.fabric.codex.cli",  # Fabric drives the Codex CLI under the hood
        resolution="preinstalled",
        settings={"sandbox": "workspace-write", "skip_git_repo_check": True, "timeout_seconds": 180},
    ),
    runtime=RuntimeConfig(mode="oneshot", transport="cli", input_schema="text", output_schema="message"),
    environment=EnvironmentConfig(provider="local"),  # the per-task workspace is set by the runtime
)

target = FabricAgentRuntime(
    config=codex_via_fabric,
    model=os.environ.get("CODEX_MODEL"),  # None → the adapter's default model
    base_dir=Path(os.environ["NEMO_FABRIC_REPO"]),  # resolves adapters/codex-cli
    work_root=OUTPUT_DIR / "fabric",
    capture_trajectory=True,  # capture the agent's ATIF trajectory as trace evidence
)
print("Target:", type(target).__name__)

> Under the hood a target is an **`AgentTaskRunner`** — any object with an async
> `run_tasks(tasks, config)` that runs each task and returns a trial. `FabricAgentRuntime` already is
> one; implementing that one method is how you'd evaluate *your own* agent (see
> [Where to go next](#next)).

<a id="run"></a>
## 6. Run the evaluation

`run_sync` runs each task through the target to produce trials, then applies each task's metrics to its
trial. It's safe inside a notebook's event loop. We pass `suite.tasks` — the tasks grouped in our
taskset.

In [ ]:
evaluator = AgentEvaluator()
result = evaluator.run_sync(
    tasks=suite.tasks,
    target=target,
    config=AgentEvalRunConfig(output_dir=OUTPUT_DIR, write_dashboard=True, parallelism=3),
)

print("run_id    :", result.run_id)
print("tasks     :", result.summary.task_count)
print("trials    :", result.summary.trial_count)
print("dashboard :", result.dashboard_path)

<a id="results"></a>
## 7. Read the results

The headline is the `correctness` **view**: because every task defined a view of that name, the summary
pools them into one `view.correctness` rollup — its mean is the fraction of tasks the agent got right,
across three different definitions of "right". No per-task boolean bookkeeping on our side.

In [ ]:
def aggregate(name: str):
    """Look up one aggregated score from the run summary by name."""
    return next((agg for agg in result.summary.scores.scores if agg.name == name), None)


correctness = aggregate("view.correctness")
print(f"correctness: {correctness.mean:.0%} of tasks passed  ({correctness.count} scored)")

# Numeric metric outputs aggregate too — e.g. mean pytest pass rate across the tasks that had tests.
pass_rate = aggregate("pytest_results.pass_rate")
if pass_rate is not None:
    print(f"pytest pass_rate (mean over tested tasks): {pass_rate.mean:.0%}")

# The LLM judge's doc_quality score for the write-docs task.
doc_quality = aggregate("doc_review.doc_quality")
if doc_quality is not None:
    print(f"doc_quality (LLM judge, 1-5): {doc_quality.mean:.1f}")

For the per-task, per-metric detail, everything is in `result.scores` (one record per task/metric, each
with its named outputs). Flatten it into a table:

In [ ]:
import pandas as pd

detail = pd.DataFrame(
    {"task": s.task_id, "metric": s.metric_type, "output": o.name, "value": o.value}
    for s in result.scores
    for o in s.outputs
)
detail

Because we ran through Fabric with `capture_trajectory=True`, each trial's evidence carries both the
final `workspace` **and** the agent's execution `trace` (ATIF). List what each trial captured:

In [ ]:
for trial in result.trials:
    keys = sorted(trial.evidence.descriptors) if trial.evidence else []
    print(f"{trial.task_id:12s} {trial.status:10s} evidence: {keys}")

To see **what the agent actually did** — its output, the files it changed, its step-by-step trajectory
— open the HTML dashboard at `result.dashboard_path`, or read the persisted bundle under
`result.output_dir` (`trials.jsonl`, `scores.jsonl`, `summary.json`). The trial evidence (the final
workspace and the ATIF trace) is what the metrics above opened to score each run.

<a id="next"></a>
## 8. Where to go next

- **Swap the harness.** Fabric selects the agent by `harness.adapter_id`. Point it at another adapter
  (e.g. `nvidia.fabric.hermes.cli`) to evaluate a different coding agent with the same tasks and
  metrics.
- **Evaluate your own agent.** Implement the `AgentTaskRunner` protocol — a class with one async
  `run_tasks(tasks, config)` that runs each task and returns a trial (output + workspace evidence).
  Everything else in this notebook stays the same.
- **Re-score without re-running the agent.** Pass precomputed `trials=` instead of `target=` to
  `run_sync` to apply new metrics to trials you already have.
- **Grade against the trajectory.** A metric can open the `trace` evidence (ATIF) to score *how* the
  agent worked — tool calls, retries, steps — not just its final files.
- **Add signals to a view, or weight them.** `SemanticReducer` also offers `ANY`, `MEAN`, and
  `WEIGHTED_MEAN` (with `ViewSignal(weight=...)`) — e.g. a partial-credit `correctness` from `pass_rate`.
- **Keep ground truth held out.** Anything a metric grades on — tests, reference solutions, rubrics —
  belongs in `reference`, overlaid or checksummed at scoring time, never in the agent's workspace.
- **Decide pass/fail in your app.** The evaluator reports scores; thresholds and gating belong to your
  CI/release process, not the evaluation itself.